In [1]:
import pandas as pd
import numpy as np
import json
import joblib
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm


In [2]:
# --- 1. Define Paths to Artifacts ---
# These paths point to the input directories from your attached Kaggle Datasets.
MODEL_PATH = '/kaggle/input/finalmodel/final-deberta-v3-model'
MLB_PATH = '/kaggle/input/finalmodel/mlb.joblib'
CONFUSION_DICT_PATH = '/kaggle/input/confusion/confusion_dictionary.json'
TEST_DATA_PATH = '/kaggle/input/map-charting-student-math-misunderstandings/test.csv'
mod = '/kaggle/input/deberta-v3-base-offline-files/deberta-v3-base-offline'

In [3]:
tokenizer = AutoTokenizer.from_pretrained(mod)
model = AutoModelForSequenceClassification.from_pretrained(mod)

2025-09-12 15:15:17.626442: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757690117.824423      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757690117.883525      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
mlb = joblib.load(MLB_PATH)
labels = mlb.classes_

In [5]:
with open(CONFUSION_DICT_PATH, 'r') as f:
    confusion_dict = json.load(f)


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval() # Set model to evaluation mode

print("Artifacts loaded successfully.")

Artifacts loaded successfully.


In [7]:
def predict_with_boost(text, row_id_debug=None):
    """
    Predicts top-3 labels using model probabilities.
    Uses confusion_dict only if predictions are duplicates or low quality.
    """
    # Tokenize
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Get logits
    with torch.no_grad():
        logits = model(**inputs).logits[0]

    probs = torch.sigmoid(logits).cpu().numpy()
    ranked_indices = np.argsort(probs)[::-1]

    # Step 1: Take top-3 labels directly from model
    final_predictions = [labels[i] for i in ranked_indices[:3]]

    # Step 2: If duplicates or weird labels, expand using confusion_dict
    if len(set(final_predictions)) < 3 or any("nan" in p.lower() for p in final_predictions):
        top1 = final_predictions[0]
        confusion_candidates = confusion_dict.get(top1, [])
        for cand in confusion_candidates:
            if len(final_predictions) >= 3:
                break
            if cand not in final_predictions:
                final_predictions.append(cand)

    # Step 3: Safety net
    while len(final_predictions) < 3:
        final_predictions.append("False_Neither:NA")

    # Debugging for first few rows
    if row_id_debug is not None and row_id_debug < 5:
        top1_conf = probs[ranked_indices[0]]
        print(f"[Row {row_id_debug}] Top1={final_predictions[0]} (conf={top1_conf:.3f})")
        print(f" → Final predictions: {final_predictions}")

    return final_predictions[:3]


In [8]:
print("Loading and preparing test data...")
test_df = pd.read_csv(TEST_DATA_PATH)


Loading and preparing test data...


In [9]:
test_df['input_text'] = test_df.apply(
    lambda row: f"Question: {row.QuestionText} Answer: {row.MC_Answer} Explanation: {row.StudentExplanation}", 
    axis=1
)

print("Generating predictions for the test set...")
all_predictions = []
# Use tqdm for a progress bar
for text in tqdm(test_df['input_text'].tolist(), desc="Predicting"):
    top_3_labels = predict_with_boost(text)
    all_predictions.append(top_3_labels)


Generating predictions for the test set...


Predicting:   0%|          | 0/3 [00:00<?, ?it/s]

In [10]:
print("Formatting and saving submission file...")

# The competition requires the ID and a space-separated string of the 3 labels
submission_df = pd.DataFrame({
    'row_id': test_df['row_id'],
    'Category:Misconception': [' '.join(pred_list) for pred_list in all_predictions]
})

# Save to submission.csv
submission_df.to_csv('submission.csv', index=False)

print("="*50)
print("submission.csv created successfully!")
print("="*50)
print(submission_df.head())

Formatting and saving submission file...
submission.csv created successfully!
   row_id                             Category:Misconception
0   36696  False_Correct:nan False_Misconception:Adding_a...
1   36697  False_Correct:nan False_Misconception:Adding_a...
2   36698  False_Correct:nan False_Misconception:Adding_a...
